# Person 2 — K-Means Clustering & Algorithm Comparison

**Goal:** Cluster trips using K-Means with the same feature space as Person 1's DBSCAN, then produce a side-by-side comparison.

**Features used for clustering (must match Person 1):**
- `pickup_lat`, `pickup_lon`
- `hour_sin`, `hour_cos` (cyclic encoding of pickup hour)

**Outputs handed off to Person 3:**
- `df_clustered` with a `kmeans_cluster` column
- `kmeans_summary` dict: `k`, `inertia`, `silhouette_score`, `davies_bouldin`
- `comparison_df` table: both algorithms side-by-side

**Workflow:**
1. Load same 10k-row subset (same `random_state=42`)  
2. Feature engineering (same as Person 1)  
3. Elbow + silhouette → choose K  
4. Run K-Means, report metrics  
5. Load Person 1's DBSCAN labels and compare  
6. Visualize both on map side-by-side

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

plt.rcParams["figure.dpi"] = 120
RANDOM_STATE = 42

## 1. Load Data (same 10k-row subset as Person 1)

In [ ]:
DATA_PATH = "../Data/FHVTrip_VehicleEmissions_Merged_2023.csv"
SUBSET_N  = 10_000  # must match Person 1

df_full = pd.read_csv(DATA_PATH, low_memory=False)
df      = df_full.sample(n=SUBSET_N, random_state=RANDOM_STATE).reset_index(drop=True)

df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], errors="coerce")
df["pickup_lat"]      = pd.to_numeric(df["pickup_lat"],      errors="coerce")
df["pickup_lon"]      = pd.to_numeric(df["pickup_lon"],      errors="coerce")
df = df.dropna(subset=["pickup_lat", "pickup_lon", "pickup_datetime"])
print(f"Working subset: {len(df):,} rows")

## 2. Feature Engineering (identical to Person 1)

In [ ]:
df["pickup_hour"] = df["pickup_datetime"].dt.hour
df["hour_sin"]    = np.sin(2 * np.pi * df["pickup_hour"] / 24)
df["hour_cos"]    = np.cos(2 * np.pi * df["pickup_hour"] / 24)

features = ["pickup_lat", "pickup_lon", "hour_sin", "hour_cos"]
scaler   = StandardScaler()
X        = scaler.fit_transform(df[features].values)

print(f"Feature matrix shape: {X.shape}")

## 3. Elbow Method + Silhouette Scores — Choose K

In [ ]:
k_range     = range(2, 21)
inertias    = []
silhouettes = []
dbi_scores  = []

for k in k_range:
    km     = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels, sample_size=5000, random_state=RANDOM_STATE))
    dbi_scores.append(davies_bouldin_score(X, labels))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(k_range, inertias, marker="o")
axes[0].set_title("Elbow: Inertia vs K")
axes[0].set_xlabel("K"); axes[0].set_ylabel("Inertia")

axes[1].plot(k_range, silhouettes, marker="o", color="green")
axes[1].set_title("Silhouette Score vs K (higher = better)")
axes[1].set_xlabel("K"); axes[1].set_ylabel("Silhouette")

axes[2].plot(k_range, dbi_scores, marker="o", color="orange")
axes[2].set_title("Davies-Bouldin Index vs K (lower = better)")
axes[2].set_xlabel("K"); axes[2].set_ylabel("DBI")

plt.tight_layout()
plt.savefig("kmeans_elbow.png", bbox_inches="tight")
plt.show()

# TODO: pick K_BEST below based on the above plots
K_BEST = silhouettes.index(max(silhouettes)) + 2   # auto-select best silhouette
print(f"Auto-selected K = {K_BEST} (highest silhouette)")

## 4. Run K-Means with Chosen K

In [ ]:
K = K_BEST   # override here with your visually chosen value if desired

km = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10)
df["kmeans_cluster"] = km.fit_predict(X)

sil_km   = silhouette_score(X, df["kmeans_cluster"], sample_size=5000, random_state=RANDOM_STATE)
dbi_km   = davies_bouldin_score(X, df["kmeans_cluster"])
inertia  = km.inertia_

print(f"K={K}")
print(f"  Inertia          : {inertia:,.1f}")
print(f"  Silhouette score : {sil_km:.4f}")
print(f"  Davies-Bouldin   : {dbi_km:.4f}")

## 5. Load DBSCAN Labels & Compare

Load the output from Person 1 and build a single comparison table.

In [ ]:
# Load Person 1's labeled output (must exist)
dbscan_df = pd.read_csv("../Data/trips_dbscan_labeled.csv")

dbscan_n_clusters = dbscan_df["dbscan_cluster"].nunique() - (1 if -1 in dbscan_df["dbscan_cluster"].values else 0)
dbscan_noise_pct  = (dbscan_df["dbscan_cluster"] == -1).mean() * 100

mask_db   = dbscan_df["dbscan_cluster"] != -1
X_db_raw  = dbscan_df.loc[mask_db, ["pickup_lat", "pickup_lon", "hour_sin", "hour_cos"]].values
X_db      = StandardScaler().fit_transform(X_db_raw)
sil_db    = silhouette_score(X_db, dbscan_df.loc[mask_db, "dbscan_cluster"], sample_size=5000, random_state=RANDOM_STATE)
dbi_db    = davies_bouldin_score(X_db, dbscan_df.loc[mask_db, "dbscan_cluster"])

comparison_df = pd.DataFrame([
    {"algorithm": "DBSCAN",  "n_clusters": dbscan_n_clusters, "noise_pct": round(dbscan_noise_pct, 1), "silhouette": round(sil_db, 4), "davies_bouldin": round(dbi_db, 4)},
    {"algorithm": "K-Means", "n_clusters": K,                 "noise_pct": 0.0,                        "silhouette": round(sil_km, 4), "davies_bouldin": round(dbi_km, 4)},
])

print("\n=== Algorithm Comparison ===")
print(comparison_df.to_string(index=False))

## 6. Side-by-Side Map Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), sharex=True, sharey=True)

# DBSCAN
noise_m = dbscan_df["dbscan_cluster"] == -1
ax1.scatter(dbscan_df.loc[noise_m, "pickup_lon"],  dbscan_df.loc[noise_m, "pickup_lat"],  c="lightgrey", s=4, alpha=0.3, label="Noise")
ax1.scatter(dbscan_df.loc[~noise_m, "pickup_lon"], dbscan_df.loc[~noise_m, "pickup_lat"], c=dbscan_df.loc[~noise_m, "dbscan_cluster"], cmap="tab20", s=5, alpha=0.7)
ax1.set_title(f"DBSCAN — {dbscan_n_clusters} clusters, {dbscan_noise_pct:.1f}% noise")
ax1.set_xlabel("Longitude"); ax1.set_ylabel("Latitude")
ax1.legend(loc="upper right")

# K-Means
sc2 = ax2.scatter(df["pickup_lon"], df["pickup_lat"], c=df["kmeans_cluster"], cmap="tab20", s=5, alpha=0.7)
ax2.set_title(f"K-Means — K={K}, no noise")
ax2.set_xlabel("Longitude")
plt.colorbar(sc2, ax=ax2, label="Cluster ID")

plt.suptitle("Clustering Comparison: Pickup Locations", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("clustering_comparison.png", bbox_inches="tight")
plt.show()

## 7. Export for Person 3

In [ ]:
df.to_csv("../Data/trips_kmeans_labeled.csv", index=False)
comparison_df.to_csv("../Data/clustering_comparison.csv", index=False)
print(f"Saved trips_kmeans_labeled.csv ({len(df):,} rows)")
print("Saved clustering_comparison.csv")